# Module 1: Building a Single Agent with Role-Based Access Control

## Overview

In this module, you'll build a **single Product Catalog Agent** with role-based access control (RBAC) using the Strands Agent SDK and **MCP (Model Context Protocol)** tools.

### Why Start with a Single Agent?

Before building complex multi-agent systems, it's critical to:

1. **Validate tool calling** - Ensure the agent correctly selects and uses the right tools
2. **Test access control** - Verify that role-based restrictions work properly
3. **Evaluate effectiveness** - Measure a single agent's quality before adding orchestration complexity

### What You'll Build

```
┌──────────────────────────────────┐
│      User Request               │
│  (with role: customer or admin)  │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   PRODUCT CATALOG AGENT          │
│   (Claude Haiku 4.5)             │
│   • Role-aware system prompt     │
│   • Tool filtering by role       │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│   Product MCP Server (FastMCP)   │
│                                  │
│  READ tools (customer + admin):  │
│  • search_products               │
│  • get_product_details           │
│  • check_inventory               │
│  • get_product_recommendations   │
│  • compare_products              │
│  • get_return_policy             │
│                                  │
│  WRITE tools (admin only):       │
│  • create_product                │
│  • update_product                │
│  • delete_product                │
│  • update_inventory              │
│  • update_pricing                │
└──────────────┬───────────────────┘
               │
┌──────────────▼───────────────────┐
│       DynamoDB Products Table    │
└──────────────────────────────────┘
```

### Learning Objectives

1. Build an agent with Strands SDK connected to an MCP tool server
2. Understand MCP architecture: agent ↔ MCP client ↔ MCP server ↔ DynamoDB
3. Implement role-based access control by filtering tools per user role
4. Test different personas (customer vs admin) against the same agent
5. Validate access control boundaries


## Step 1: Environment Setup

First, let's set up our environment and verify we have access to the required resources.


In [1]:
# Install dependencies if needed
!pip install -q strands-agents strands-agents-tools boto3 mcp

In [2]:
import boto3
import json
import os
import sys
from datetime import datetime

# Get AWS region
session = boto3.Session()
REGION = session.region_name or "us-west-2"
print(f"AWS Region: {REGION}")

# Set environment variables for tools
os.environ["AWS_REGION"] = REGION

# Get table names from SSM (set by workshop infrastructure)
ssm = boto3.client("ssm", region_name=REGION)

try:
    os.environ["PRODUCTS_TABLE_NAME"] = ssm.get_parameter(
        Name="ecommerce-workshop-products-table"
    )["Parameter"]["Value"]
    print(f"Products Table: {os.environ['PRODUCTS_TABLE_NAME']}")
except Exception as e:
    print(f"Note: Could not retrieve SSM parameters ({e})")
    print("Using default table names - ensure infrastructure is deployed")
    os.environ["PRODUCTS_TABLE_NAME"] = "ecommerce-workshop-products"

AWS Region: us-west-2
Products Table: ecommerce-workshop-products


## Step 2: Understanding MCP Tools

Our agent uses **MCP (Model Context Protocol)** to access tools from a separate MCP server process. This provides:

- **Decoupling**: Tools run as a separate process from the agent
- **Reusability**: MCP servers can be shared across multiple agents
- **Standardization**: Follows the MCP protocol for tool discovery and invocation

### MCP Server Architecture

```
Agent (Haiku)  ──MCPClient──►  MCP Server  ──►  DynamoDB
                  (stdio)      (FastMCP)        Products Table
```

### Tools in Our MCP Server

| Tool                          | Access Level     | Description                         |
| ----------------------------- | ---------------- | ----------------------------------- |
| `search_products`             | Customer + Admin | Search catalog by keywords/category |
| `get_product_details`         | Customer + Admin | Get full product details            |
| `check_inventory`             | Customer + Admin | Check stock availability            |
| `get_product_recommendations` | Customer + Admin | Get product recommendations         |
| `compare_products`            | Customer + Admin | Side-by-side product comparison     |
| `get_return_policy`           | Customer + Admin | Return policy information           |
| `create_product`              | **Admin only**   | Add new product to catalog          |
| `update_product`              | **Admin only**   | Modify product information          |
| `delete_product`              | **Admin only**   | Remove product (soft delete)        |
| `update_inventory`            | **Admin only**   | Adjust stock quantities             |
| `update_pricing`              | **Admin only**   | Change prices, set sales            |


In [3]:
# Verify MCP server exists
from pathlib import Path

mcp_server_path = Path("mcp_servers/product_mcp_server.py")
print(
    f"Product MCP Server: {mcp_server_path} - {'exists' if mcp_server_path.exists() else 'MISSING'}"
)

Product MCP Server: mcp_servers/product_mcp_server.py - exists


## Step 3: Build the Product Catalog Agent (No RBAC)

Let's first build a basic agent that connects to the MCP server with **all tools available** (no role filtering yet). This lets us verify the MCP connection and tool calling work correctly.


In [4]:
from strands import Agent
from strands.models import BedrockModel
from strands.tools.mcp import MCPClient
from mcp import StdioServerParameters
from mcp.client.stdio import stdio_client

# Model configuration
HAIKU_MODEL_ID = "global.anthropic.claude-haiku-4-5-20251001-v1:0"
print(f"Model: {HAIKU_MODEL_ID}")

# Connect to the product MCP server
server_params = StdioServerParameters(
    command=sys.executable,
    args=[str(mcp_server_path)],
    env={
        **os.environ,
        "AWS_REGION": REGION,
        "PRODUCTS_TABLE": os.environ.get(
            "PRODUCTS_TABLE_NAME", "ecommerce-workshop-products"
        ),
    },
)

mcp_client = MCPClient(lambda: stdio_client(server_params))
mcp_client.__enter__()

# Discover all tools from the MCP server
all_tools = mcp_client.list_tools_sync()
print(f"\nDiscovered {len(all_tools)} tools from MCP server:")
for tool in all_tools:
    print(f"  - {tool.tool_name}")

Model: global.anthropic.claude-haiku-4-5-20251001-v1:0

Discovered 11 tools from MCP server:
  - search_products
  - get_product_details
  - check_inventory
  - get_product_recommendations
  - compare_products
  - get_return_policy
  - create_product
  - update_product
  - delete_product
  - update_inventory
  - update_pricing


In [5]:
# Create a basic agent with ALL tools (no RBAC yet)
basic_agent = Agent(
    name="ProductCatalogAgent",
    model=BedrockModel(
        model_id=HAIKU_MODEL_ID, region_name=REGION, temperature=0.3, max_tokens=1500
    ),
    system_prompt="You are a Product Catalog Assistant. Help users with product queries using the available tools.",
    tools=all_tools,
)
print("Basic agent created with all 11 tools (no role filtering)")

Basic agent created with all 11 tools (no role filtering)


## Step 4: Test Basic Tool Calling

Let's verify the agent correctly uses MCP tools to answer product queries.


In [6]:
# Test 1: Product search
print("=" * 60)
print("Test 1: Product Search")
print("=" * 60)
response = basic_agent("Do you have any wireless headphones under $100?")
print(f"\nResponse: {response}")

Test 1: Product Search
I'll search for wireless headphones under $100 for you.
Tool #1: search_products
Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones** (PROD-001)
- **Price:** $79.99
- **In Stock:** Yes
- **Description:** Premium over-ear wireless headphones with 40-hour battery life, active noise cancellation, and comfortable memory foam ear cushions.

This is an excellent option at under $100! It features:
- Active noise cancellation
- 40-hour battery life
- Memory foam ear cushions for comfort
- Over-ear design

I also found another option that's slightly above your budget:
- **Noise Canceling Earbuds** (PROD-055) at $149.99 if you're interested in true wireless earbuds instead.

Would you like more details about the $79.99 headphones, or would you like to check inventory or compare them with other options?
Response: Great! I found one wireless headphone option under $100:

**Wireless Bluetooth Headphones** (PROD-001)
- **Price:** $79.99
-

In [7]:
# Test 2: Product details
print("=" * 60)
print("Test 2: Product Details")
print("=" * 60)
response = basic_agent("Tell me about the Gaming Mechanical Keyboard PROD-077")
print(f"\nResponse: {response}")

Test 2: Product Details

Tool #2: get_product_details
Here are the details for the **Gaming Mechanical Keyboard RGB** (PROD-077):

**Price:** $159.99

**Description:** Full-size mechanical gaming keyboard with hot-swappable switches, per-key RGB lighting, and detachable wrist rest.

**Key Specifications:**
- **Switch Type:** Hot-swappable (Cherry MX compatible) - allows you to easily swap out switches
- **Lighting:** Per-key RGB with 16.8M colors for customizable lighting effects
- **Connection:** USB-C detachable cable for easy portability
- **Anti-Ghosting:** Full N-key rollover for responsive gaming
- **Media Controls:** Dedicated media keys + volume wheel

**Availability:** In Stock (60 units available)

**Rating:** 4.6 out of 5 stars

**Warranty:** 2 years

**Return Policy:** 30-day return policy

This is a solid gaming keyboard with professional-grade features! The hot-swappable switches are a great feature if you like to customize your keyboard, and the full N-key rollover ensur

In [8]:
# Test 3: Inventory check
print("=" * 60)
print("Test 3: Inventory Check")
print("=" * 60)
response = basic_agent("Is the 4K webcam PROD-088 in stock?")
print(f"\nResponse: {response}")

Test 3: Inventory Check

Tool #3: check_inventory
Unfortunately, the **4K Webcam** (PROD-088) is currently **out of stock**.

**Restock Information:**
- **Expected Restock Date:** January 15, 2025

The product should be back in stock soon! Would you like me to:
1. Get more details about this 4K webcam?
2. Search for alternative webcams that are currently in stock?
3. Help you with anything else?
Response: Unfortunately, the **4K Webcam** (PROD-088) is currently **out of stock**.

**Restock Information:**
- **Expected Restock Date:** January 15, 2025

The product should be back in stock soon! Would you like me to:
1. Get more details about this 4K webcam?
2. Search for alternative webcams that are currently in stock?
3. Help you with anything else?



In [9]:
# Test 4: Product comparison
print("=" * 60)
print("Test 4: Product Comparison")
print("=" * 60)
response = basic_agent(
    "Compare the Wireless Bluetooth Headphones PROD-001 with the Noise Canceling Earbuds PROD-055"
)
print(f"\nResponse: {response}")

Test 4: Product Comparison

Tool #4: compare_products
Here's a detailed comparison of the two audio products:

| Feature | PROD-001: Wireless Bluetooth Headphones | PROD-055: Noise Canceling Earbuds |
|---------|----------------------------------------|----------------------------------|
| **Price** | $79.99 | $149.99 |
| **Type** | Over-ear | True wireless earbuds |
| **Rating** | 4.5/5 | 4.5/5 |
| **In Stock** | Yes | Yes |
| **Battery Life** | 40 hours | 8 hours (earbuds) + 24 hours (with case) |
| **Bluetooth Version** | 5.3 | 5.2 |
| **Noise Cancellation** | Active ANC | Adaptive noise cancellation |
| **Driver Size** | 40mm | N/A |
| **Weight** | 250g | N/A |
| **Water Resistance** | N/A | IPX4 |
| **Codecs** | N/A | AAC, aptX Adaptive |
| **Warranty** | 1 year | 1 year |

**Key Differences:**

**Choose PROD-001 (Wireless Bluetooth Headphones) if you want:**
- Best value at $79.99
- Exceptional 40-hour battery life
- Over-ear comfort for extended wear
- Larger 40mm drivers for po

In [10]:
# Clean up the basic agent's MCP connection
mcp_client.__exit__(None, None, None)
print("Basic agent MCP connection cleaned up")

Basic agent MCP connection cleaned up


## Step 5: Understanding Role-Based Access Control (RBAC)

Now that we've confirmed the agent and tools work, let's add **access control**.

### The Problem

Our MCP server exposes 11 tools - 6 read-only tools (safe for customers) and 5 write tools (admin-only). Without RBAC, any user could create, update, or delete products.

### The Solution: Tool Filtering

We implement RBAC by **filtering which tools the agent receives** based on the user's role:

```python
# Customer gets 6 read-only tools
customer_tools = filter_by_role(all_tools, role="customer")  # 6 tools

# Admin gets all 11 tools
admin_tools = filter_by_role(all_tools, role="admin")  # 11 tools
```

Since the LLM can only call tools it knows about, **removing a tool from the agent's tool list is an effective access control boundary**.

### Production Mapping (Module 03)

In production, user roles come from **AgentCore Identity** via JWT tokens:

- A **Cognito User Pool** manages users with `customer` and `admin` groups
- The **JWT inbound authorizer** on AgentCore Runtime validates tokens
- Agent code extracts the role from JWT claims and filters tools accordingly


In [11]:
# Add agents to path and import the RBAC-enabled agent
sys.path.insert(0, "agents")
from product_catalog_agent import (
    ProductCatalogAgent,
    UserSession,
    create_product_catalog_agent,
    CUSTOMER_TOOLS,
    ADMIN_TOOLS,
    ADMIN_ONLY_TOOLS,
    CUSTOMER_PERSONAS,
    ADMIN_PERSONAS,
)

print("RBAC Configuration:")
print(f"  Customer tools ({len(CUSTOMER_TOOLS)}): {CUSTOMER_TOOLS}")
print(f"  Admin-only tools ({len(ADMIN_ONLY_TOOLS)}): {ADMIN_ONLY_TOOLS}")
print(f"  Total admin tools ({len(ADMIN_TOOLS)}): {len(ADMIN_TOOLS)}")

RBAC Configuration:
  Customer tools (6): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy']
  Admin-only tools (5): ['create_product', 'update_product', 'delete_product', 'update_inventory', 'update_pricing']
  Total admin tools (11): 11


## Step 6: Create User Sessions (Personas)

Let's define our test personas. In production, these would come from JWT claims.


In [12]:
# Define test personas
customer_session = UserSession(
    user_id="CUST-1001",
    role="customer",
    email="john.smith@email.com",
    name="John Smith",
)

admin_session = UserSession(
    user_id="ADMIN-001",
    role="admin",
    email="alice.admin@company.com",
    name="Alice (Admin)",
)

print("Test Personas:")
print(
    f"  Customer: {customer_session.name} ({customer_session.email}) - role: {customer_session.role}"
)
print(
    f"  Admin:    {admin_session.name} ({admin_session.email}) - role: {admin_session.role}"
)

Test Personas:
  Customer: John Smith (john.smith@email.com) - role: customer
  Admin:    Alice (Admin) (alice.admin@company.com) - role: admin


## Step 7: Test Customer Persona

The customer agent should be able to search, view, and compare products, but should **NOT** be able to create, update, or delete products.


In [13]:
# Create agent with CUSTOMER role
customer_agent = create_product_catalog_agent(
    region=REGION, user_session=customer_session
)

user_info = customer_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

Agent created for: John Smith (role: customer)
Available tools (6): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy']


In [14]:
# Customer Test 1: Search (should work)
print("=" * 60)
print("Customer Test 1: Product Search (ALLOWED)")
print("=" * 60)
response = customer_agent("I'm looking for a good gaming keyboard with RGB lighting")
print(f"\nResponse: {response}")

Customer Test 1: Product Search (ALLOWED)
I'll search for gaming keyboards with RGB lighting for you.
Tool #1: search_products
Great! I found a gaming keyboard with RGB lighting. Let me get more detailed information about it for you.
Tool #2: get_product_details
Perfect! Here's what I found:

## **Gaming Mechanical Keyboard RGB** (PROD-077)
**Price:** $159.99

### Key Features:
- **Per-key RGB Lighting** - 16.8 million colors for full customization
- **Hot-swappable Switches** - Cherry MX compatible, so you can easily swap switches
- **Full-size Layout** - Complete keyboard with all standard keys
- **Full N-key Rollover** - Anti-ghosting for responsive gaming
- **Detachable Wrist Rest** - Included for comfort during long gaming sessions
- **USB-C Detachable Cable** - Easy to connect and transport
- **Dedicated Media Controls** - Media keys plus volume wheel for convenience

### Additional Info:
- **Rating:** 4.6 out of 5 stars
- **Warranty:** 2 years
- **Return Policy:** 30-day return 

In [15]:
# Customer Test 2: Recommendations (should work)
print("=" * 60)
print("Customer Test 2: Recommendations (ALLOWED)")
print("=" * 60)
response = customer_agent("Can you recommend some audio products for a music lover?")
print(f"\nResponse: {response}")

Customer Test 2: Recommendations (ALLOWED)
I'll get some audio product recommendations for a music lover.
Tool #3: get_product_recommendations
Excellent! Here are my top audio product recommendations for a music lover:

## 1. **Wireless Bluetooth Headphones** (PROD-001)
**Price:** $79.99
- Premium over-ear design with memory foam ear cushions for comfort
- Active noise cancellation to immerse yourself in music
- Impressive 40-hour battery life
- Great value for the price!

## 2. **Noise Canceling Earbuds** (PROD-055)
**Price:** $149.99
- True wireless with adaptive noise cancellation
- Transparency mode to hear your surroundings when needed
- 8-hour battery per charge (with charging case for extended use)
- Perfect for on-the-go listening

## 3. **Portable Bluetooth Speaker** (PROD-033)
**Price:** $199.99
- 360-degree sound for immersive audio experience
- Waterproof design - great for any environment
- Impressive 20-hour battery life
- Party mode allows pairing multiple speakers toget

In [16]:
# Customer Test 3: Attempt admin action (should be REFUSED)
print("=" * 60)
print("Customer Test 3: Create Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent(
    "Create a new product called 'Super Speaker' with price $299.99 in the Audio category"
)
print(f"\nResponse: {response}")

Customer Test 3: Create Product (SHOULD BE REFUSED)
I appreciate the request, but I'm unable to create, update, delete, or modify products in the catalog. Those are admin-only actions that require special permissions.

As a Product Catalog Assistant, I can only:
- ✅ Search for existing products
- ✅ View product details and specifications
- ✅ Check inventory and availability
- ✅ Compare products
- ✅ Provide recommendations
- ✅ Explain return policies

To add a new product to the catalog, you would need to contact an administrator or someone with catalog management access. They would have the tools and permissions to create new product listings.

Is there anything else I can help you with regarding our current product catalog? I'd be happy to search for speakers or other audio products, or help you find the perfect product for your needs!
Response: I appreciate the request, but I'm unable to create, update, delete, or modify products in the catalog. Those are admin-only actions that requ

In [17]:
# Customer Test 4: Attempt to delete a product (should be REFUSED)
print("=" * 60)
print("Customer Test 4: Delete Product (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Delete the product PROD-001 from the catalog")
print(f"\nResponse: {response}")

Customer Test 4: Delete Product (SHOULD BE REFUSED)
I'm unable to delete products from the catalog. That's an admin-only action that requires special permissions.

As a Product Catalog Assistant, I can only help with:
- ✅ Searching and viewing products
- ✅ Getting detailed product information
- ✅ Checking inventory and availability
- ✅ Comparing products
- ✅ Providing recommendations
- ✅ Explaining return policies

**Catalog modifications** (creating, updating, or deleting products) require administrator access.

If you need to remove or modify a product listing, please contact your catalog administrator or management team.

Is there anything else I can help you with regarding our current products? For example, I could help you find alternatives to PROD-001 or search for other products you're interested in!
Response: I'm unable to delete products from the catalog. That's an admin-only action that requires special permissions.

As a Product Catalog Assistant, I can only help with:
- ✅ S

In [18]:
# Clean up customer agent
customer_agent.cleanup()
print("Customer agent cleaned up")

Customer agent cleaned up


## Step 8: Test Admin Persona

The admin agent should have **full access** to all tools, including creating, updating, and deleting products.


In [19]:
# Create agent with ADMIN role
admin_agent = create_product_catalog_agent(region=REGION, user_session=admin_session)

user_info = admin_agent.get_user_info()
print(f"Agent created for: {user_info['name']} (role: {user_info['role']})")
print(f"Available tools ({user_info['tools_available']}): {user_info['tools']}")

Agent created for: Alice (Admin) (role: admin)
Available tools (11): ['search_products', 'get_product_details', 'check_inventory', 'get_product_recommendations', 'compare_products', 'get_return_policy', 'create_product', 'update_product', 'delete_product', 'update_inventory', 'update_pricing']


In [20]:
# Admin Test 1: Search still works (admin has all customer tools too)
print("=" * 60)
print("Admin Test 1: Product Search (ALLOWED)")
print("=" * 60)
response = admin_agent("Search for all audio products")
print(f"\nResponse: {response}")

Admin Test 1: Product Search (ALLOWED)

Tool #1: search_products
The search returned no results for audio products in the catalog. This could mean:

1. **No audio products currently exist** in the catalog
2. **The catalog may be empty** or newly initialized

Would you like me to:
- **Create new audio products** for the catalog? (I can add headphones, speakers, microphones, etc.)
- **Search for products in a different category** to see what's available?
- **Search with different keywords** to verify the catalog status?

Please let me know how you'd like to proceed!
Response: The search returned no results for audio products in the catalog. This could mean:

1. **No audio products currently exist** in the catalog
2. **The catalog may be empty** or newly initialized

Would you like me to:
- **Create new audio products** for the catalog? (I can add headphones, speakers, microphones, etc.)
- **Search for products in a different category** to see what's available?
- **Search with different k

In [21]:
# Admin Test 2: Create a new product
print("=" * 60)
print("Admin Test 2: Create Product (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Create a new product with ID PROD-200 called 'Premium Gaming Headset' "
    "in the Audio category for $129.99. Description: 'Professional gaming headset "
    "with 7.1 surround sound, detachable microphone, and RGB lighting.' "
    "Specifications: bluetooth 5.3, weight 320g, driver size 50mm. "
    "Set initial stock to 50 units."
)
print(f"\nResponse: {response}")

Admin Test 2: Create Product (ALLOWED)

Tool #2: create_product
✅ **Product Created Successfully!**

**Product Details:**
- **Product ID:** PROD-200
- **Name:** Premium Gaming Headset
- **Category:** Audio
- **Price:** $129.99
- **Description:** Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.
- **Specifications:**
  - Bluetooth: 5.3
  - Weight: 320g
  - Driver Size: 50mm
- **Initial Stock:** 50 units
- **Warranty:** 1 year manufacturer warranty (default)
- **Return Policy:** 30-day return policy (default)

The product is now active in the catalog and ready for sale!
Response: ✅ **Product Created Successfully!**

**Product Details:**
- **Product ID:** PROD-200
- **Name:** Premium Gaming Headset
- **Category:** Audio
- **Price:** $129.99
- **Description:** Professional gaming headset with 7.1 surround sound, detachable microphone, and RGB lighting.
- **Specifications:**
  - Bluetooth: 5.3
  - Weight: 320g
  - Driver Size: 50mm
- **Initial Sto

In [22]:
# Admin Test 3: Update pricing with a sale
print("=" * 60)
print("Admin Test 3: Update Pricing (ALLOWED)")
print("=" * 60)
response = admin_agent(
    "Set a sale price of $99.99 for PROD-200 (regular price stays $129.99) "
    "until 2025-03-31"
)
print(f"\nResponse: {response}")

Admin Test 3: Update Pricing (ALLOWED)

Tool #3: update_pricing
✅ **Pricing Updated Successfully!**

**PROD-200 - Premium Gaming Headset**

**Pricing Details:**
- **Regular Price:** $129.99 (unchanged)
- **Sale Price:** $99.99
- **Discount:** 23.1% off
- **Sale End Date:** 2025-03-31

Customers will see the sale price of $99.99 until March 31, 2025, after which the regular price of $129.99 will apply automatically.
Response: ✅ **Pricing Updated Successfully!**

**PROD-200 - Premium Gaming Headset**

**Pricing Details:**
- **Regular Price:** $129.99 (unchanged)
- **Sale Price:** $99.99
- **Discount:** 23.1% off
- **Sale End Date:** 2025-03-31

Customers will see the sale price of $99.99 until March 31, 2025, after which the regular price of $129.99 will apply automatically.



In [23]:
# Admin Test 4: Update inventory
print("=" * 60)
print("Admin Test 4: Update Inventory (ALLOWED)")
print("=" * 60)
response = admin_agent("Set the inventory for PROD-088 (4K webcam) to 100 units")
print(f"\nResponse: {response}")

Admin Test 4: Update Inventory (ALLOWED)

Tool #4: update_inventory
✅ **Inventory Updated Successfully!**

**PROD-088 - Webcam 4K HDR**

**Inventory Details:**
- **New Stock Quantity:** 100 units
- **Status:** In Stock
- **Product ID:** PROD-088

The inventory for the 4K webcam has been updated and is now ready for sale.
Response: ✅ **Inventory Updated Successfully!**

**PROD-088 - Webcam 4K HDR**

**Inventory Details:**
- **New Stock Quantity:** 100 units
- **Status:** In Stock
- **Product ID:** PROD-088

The inventory for the 4K webcam has been updated and is now ready for sale.



In [24]:
# Admin Test 5: Delete (discontinue) a product
print("=" * 60)
print("Admin Test 5: Delete Product (ALLOWED - soft delete)")
print("=" * 60)
response = admin_agent("Discontinue product PROD-200")
print(f"\nResponse: {response}")

Admin Test 5: Delete Product (ALLOWED - soft delete)

Tool #5: delete_product
✅ **Product Discontinued Successfully!**

**PROD-200 - Premium Gaming Headset**

**Discontinuation Details:**
- **Action:** Soft Delete (Discontinued)
- **Status:** Removed from active catalog
- **Record Retention:** Product record is retained for order history and audit purposes

**Important Note:** This is a soft delete, which means:
- The product is no longer visible to customers or in active inventory
- Historical orders containing this product remain intact
- The product record is preserved in the system for compliance and audit trails
- The product can be reactivated if needed in the future
Response: ✅ **Product Discontinued Successfully!**

**PROD-200 - Premium Gaming Headset**

**Discontinuation Details:**
- **Action:** Soft Delete (Discontinued)
- **Status:** Removed from active catalog
- **Record Retention:** Product record is retained for order history and audit purposes

**Important Note:** This i

In [25]:
# Clean up admin agent
admin_agent.cleanup()
print("Admin agent cleaned up")

Admin agent cleaned up


## Step 9: Access Control Boundary Test

Let's run a cross-role validation to confirm the RBAC boundary holds:

1. Admin creates a product → Customer can find it via search
2. Customer cannot modify the product the admin created


In [26]:
# Create both agents
admin_agent = create_product_catalog_agent(region=REGION, user_session=admin_session)
customer_agent = create_product_catalog_agent(
    region=REGION, user_session=customer_session
)

print(f"Admin tools:    {admin_agent.get_user_info()['tools_available']} tools")
print(f"Customer tools: {customer_agent.get_user_info()['tools_available']} tools")

Admin tools:    11 tools
Customer tools: 6 tools


In [27]:
# Step A: Admin creates a new product
print("=" * 60)
print("Step A: Admin creates product PROD-300")
print("=" * 60)
response = admin_agent(
    "Create product PROD-300 called 'Wireless Charging Pad' in Accessories category "
    "for $39.99. Description: 'Fast wireless charger compatible with all Qi devices, "
    "supports up to 15W charging.' Specs: charging speed 15W, compatibility Qi, weight 120g. "
    "Stock: 200 units."
)
print(f"\nAdmin Response: {response}")

Step A: Admin creates product PROD-300

Tool #1: create_product
Perfect! ✅ **Product Created Successfully**

**Product Details:**
- **Product ID:** PROD-300
- **Name:** Wireless Charging Pad
- **Category:** Accessories
- **Price:** $39.99
- **Description:** Fast wireless charger compatible with all Qi devices, supports up to 15W charging.
- **Specifications:**
  - Charging Speed: 15W
  - Compatibility: Qi
  - Weight: 120g
- **Initial Stock:** 200 units
- **Warranty:** 1 year manufacturer warranty (default)
- **Return Policy:** 30-day return policy (default)

The product is now active in the catalog and ready for customer purchases!
Admin Response: Perfect! ✅ **Product Created Successfully**

**Product Details:**
- **Product ID:** PROD-300
- **Name:** Wireless Charging Pad
- **Category:** Accessories
- **Price:** $39.99
- **Description:** Fast wireless charger compatible with all Qi devices, supports up to 15W charging.
- **Specifications:**
  - Charging Speed: 15W
  - Compatibility: Qi

In [28]:
# Step B: Customer can find the new product
print("=" * 60)
print("Step B: Customer searches for the new product")
print("=" * 60)
response = customer_agent("Do you have any wireless charging pads?")
print(f"\nCustomer Response: {response}")

Step B: Customer searches for the new product
I'll search for wireless charging pads in our catalog.
Tool #1: search_products
Great! We do have wireless charging pads. Here's what I found:

**Wireless Charging Pad** (PROD-300)
- **Price:** $39.99
- **Category:** Accessories
- **Status:** ✓ In Stock
- **Description:** Fast wireless charger compatible with all Qi devices, supports up to 15W charging

This is a great option if you're looking for a reliable wireless charging solution. It's compatible with any Qi-enabled device (most modern smartphones, earbuds, and smartwatches).

Would you like me to:
1. Get more detailed specifications about this charging pad?
2. Check current inventory levels?
3. Search for other charging accessories or related products?
4. Help you find a device to use with this charger?
Customer Response: Great! We do have wireless charging pads. Here's what I found:

**Wireless Charging Pad** (PROD-300)
- **Price:** $39.99
- **Category:** Accessories
- **Status:** ✓ 

In [29]:
# Step C: Customer tries to modify the product (SHOULD FAIL)
print("=" * 60)
print("Step C: Customer tries to change the price (SHOULD BE REFUSED)")
print("=" * 60)
response = customer_agent("Change the price of PROD-300 to $19.99")
print(f"\nCustomer Response: {response}")

Step C: Customer tries to change the price (SHOULD BE REFUSED)
I appreciate you asking, but I'm unable to modify product information like prices. That's an admin-only function that requires special permissions.

To change the price of PROD-300 (Wireless Charging Pad), you would need to:
- Contact a store administrator or manager
- Access the admin dashboard (if you have admin credentials)
- Submit a request through the appropriate channels in your organization

Is there anything else I can help you with regarding this product, such as getting more details, checking inventory, or finding similar products?
Customer Response: I appreciate you asking, but I'm unable to modify product information like prices. That's an admin-only function that requires special permissions.

To change the price of PROD-300 (Wireless Charging Pad), you would need to:
- Contact a store administrator or manager
- Access the admin dashboard (if you have admin credentials)
- Submit a request through the appropria

In [30]:
# Step D: Clean up - admin deletes the test product
print("=" * 60)
print("Step D: Admin cleans up test product")
print("=" * 60)
response = admin_agent("Delete product PROD-300")
print(f"\nAdmin Response: {response}")

Step D: Admin cleans up test product

Tool #2: delete_product
✅ **Product Deleted Successfully**

**Deletion Details:**
- **Product ID:** PROD-300
- **Product Name:** Wireless Charging Pad
- **Action Type:** Soft Delete (Discontinued)
- **Status:** Removed from active catalog

**Important Note:** This is a **soft delete**, which means:
- The product has been marked as "discontinued" and is no longer visible in the active catalog
- The product record is retained in the system for order history and audit purposes
- Customers cannot purchase this product going forward
- The product data can be referenced for historical orders if needed

If you need to permanently remove all traces of this product, please let me know and we can discuss archival options.
Admin Response: ✅ **Product Deleted Successfully**

**Deletion Details:**
- **Product ID:** PROD-300
- **Product Name:** Wireless Charging Pad
- **Action Type:** Soft Delete (Discontinued)
- **Status:** Removed from active catalog

**Import

In [31]:
# Clean up both agents
admin_agent.cleanup()
customer_agent.cleanup()
print("Both agents cleaned up")

Both agents cleaned up


## Step 10: From Prototype to Production — AgentCore Identity

In this module, we simulated RBAC using a local `UserSession` class. In **Module 3** (Production Deployment), this maps to real authentication using **Amazon Bedrock AgentCore Identity**:

### Production Architecture

```
┌────────────┐     JWT Token      ┌─────────────────────┐
│  Web App   │ ──────────────────► │  AgentCore Runtime  │
│  (React)   │                     │                     │
└─────┬──────┘                     │  JWT Authorizer     │
      │                            │  validates token    │
      │ Login                      │                     │
      ▼                            │  Agent code reads   │
┌─────────────┐                    │  role from claims   │
│  Cognito    │                    │                     │
│  User Pool  │                    │  Tools filtered     │
│             │                    │  by role            │
│ Groups:     │                    └─────────────────────┘
│ - customer  │
│ - admin     │
└─────────────┘
```

### Key Mapping

| Prototype (Module 01) | Production (Module 03)                 |
| --------------------- | -------------------------------------- |
| `UserSession` class   | JWT token claims                       |
| `role` field          | Cognito group membership               |
| `user_id` field       | JWT `sub` claim                        |
| `email` field         | JWT `email` claim                      |
| Local tool filtering  | Same logic, but role from verified JWT |

### Why This Pattern Works

1. **Defense in depth**: Tool filtering prevents the LLM from even knowing about restricted tools
2. **System prompt reinforcement**: Role-aware prompts add a second layer of guidance
3. **Verifiable**: You can inspect which tools an agent received at construction time
4. **Standard OAuth/JWT**: AgentCore Identity uses industry-standard authentication


## Summary

In this module, you built a **single Product Catalog Agent** with role-based access control that:

1. **Connects to an MCP server** with 11 tools (6 read + 5 write)
2. **Filters tools by role** — customers get 6 read tools, admins get all 11
3. **Uses role-aware system prompts** — different instructions per persona
4. **Validates access control** — customer cannot perform admin operations

### Key Takeaways

- **Start simple**: Validate one agent thoroughly before building multi-agent systems
- **RBAC via tool filtering**: Removing tools from the agent is a strong access control boundary
- **Persona testing**: Same agent code, different behaviors based on role
- **Production-ready pattern**: Local prototype maps directly to AgentCore Identity JWT auth

### Key Files

| File                                | Description                                                 |
| ----------------------------------- | ----------------------------------------------------------- |
| `mcp_servers/product_mcp_server.py` | MCP server with 11 product tools (6 read + 5 admin)         |
| `agents/product_catalog_agent.py`   | Agent with RBAC (UserSession, tool filtering, role prompts) |

### Next Steps

In **Module 2**, we'll evaluate this agent with a comprehensive test dataset to establish baseline metrics for quality, tool accuracy, and access control compliance.


In [32]:
# Save region for use in next module
%store REGION
print("Session data saved for Module 2!")

Stored 'REGION' (str)
Session data saved for Module 2!
